# **Часть 1. Знакомство с данными в PostgreSQL**


#### **Задание 1.1: загрузите и изучите данные**


Перед запуском: установите зависимости из `requirements.txt` в локальном окружении и заполните `.env` по `.env.example`. Установка выполняется один раз из корня проекта: `python -m pip install -r requirements.txt`.


In [1]:
import os
import sys
from pathlib import Path
from io import BytesIO
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import boto3
from dotenv import load_dotenv
from botocore.config import Config
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Работает при запуске из корня проекта или notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'dags' / 'calculate_batch_features.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Откройте тетрадь из папки проекта')
sys.path.insert(0, str(PROJECT_ROOT / 'dags'))
import calculate_batch_features as batch_module

load_dotenv(PROJECT_ROOT / '.env', override=False)
required_env = ['DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']
missing = [name for name in required_env
           if not os.getenv(name) or os.getenv(name).startswith('<')]
if missing:
    raise ValueError(f'Заполните .env: {missing}')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
connection_url = URL.create(
    'postgresql+psycopg2',
    username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    database=os.environ['DB_NAME'],
)
# URL хранится в памяти; не выводим его и секреты в тетрадь.
engine = create_engine(
    connection_url,
    connect_args={'sslmode': 'require', 'connect_timeout': 15},
    pool_pre_ping=True,
)
with engine.connect() as connection:
    connection.execute(text('SELECT 1'))
print('Подключение к PostgreSQL установлено.')


Подключение к PostgreSQL установлено.


In [2]:
# Полная история нужна только для исследования исходных данных.
# В DAG используется оконная загрузка без full_history=True.
raw_tables = batch_module.load_source_tables(
    postgres_uri=connection_url,
    full_history=True,
)
for name, df in raw_tables.items():
    print(f'{name}: {df.shape}')
customers_raw = raw_tables['customers']
sessions_raw = raw_tables['sessions']
events_raw = raw_tables['events']
orders_raw = raw_tables['orders']


customers: (17846, 2)
sessions: (37699, 3)
events: (239089, 5)
orders: (10652, 4)


In [3]:
for name, df in raw_tables.items():
    print(f'\nТаблица {name}')
    df.info()
    display(df.head())


Таблица customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17846 entries, 0 to 17845
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  17846 non-null  int64 
 1   signup_date  17846 non-null  object
dtypes: int64(1), object(1)
memory usage: 279.0+ KB


,customer_id,signup_date
0,1,2020-09-04
1,2,2020-04-05
2,3,2023-08-31
3,4,2022-06-30
4,5,2022-07-22



Таблица sessions
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37699 entries, 0 to 37698
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   session_id   37699 non-null  int64         
 1   customer_id  37699 non-null  int64         
 2   start_time   37699 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(2)
memory usage: 883.7 KB


,session_id,customer_id,start_time
0,2,13917,2025-01-31 21:29:42
1,3,1022,2024-02-19 00:52:50
2,4,2882,2024-08-04 19:54:31
3,9,6476,2024-02-22 16:57:44
4,10,6145,2024-12-04 18:53:13



Таблица events
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239089 entries, 0 to 239088
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   event_id    239089 non-null  int64         
 1   session_id  239089 non-null  int64         
 2   timestamp   239089 non-null  datetime64[ns]
 3   event_type  239089 non-null  object        
 4   product_id  214208 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 9.1+ MB


,event_id,session_id,timestamp,event_type,product_id
0,11,2,2025-01-31 21:48:42,page_view,929.0
1,12,2,2025-01-31 21:57:42,page_view,226.0
2,13,2,2025-01-31 21:59:07,add_to_cart,226.0
3,14,2,2025-01-31 22:01:42,page_view,864.0
4,15,2,2025-01-31 22:25:42,page_view,1151.0



Таблица orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10652 entries, 0 to 10651
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     10652 non-null  int64         
 1   customer_id  10652 non-null  int64         
 2   order_time   10652 non-null  datetime64[ns]
 3   total_usd    10652 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 333.0 KB


,order_id,customer_id,order_time,total_usd
0,1,13917,2025-01-31 23:07:42,85.72
1,2,1022,2024-02-19 01:17:50,116.17
2,3,6145,2024-12-04 20:24:13,137.35
3,4,3152,2024-07-17 08:50:47,27.35
4,11,14044,2025-03-12 02:48:39,47.40


In [4]:
primary_keys = {
    'customers': 'customer_id',
    'sessions': 'session_id',
    'events': 'event_id',
    'orders': 'order_id',
}

summary_rows = []

for name, df in raw_tables.items():
    key = primary_keys[name]
    valid_keys = df[key].dropna()

    summary_rows.append({
        'table': name,
        'rows': len(df),
        'primary_key': key,
        'unique_keys': valid_keys.nunique(),
        'missing_keys': df[key].isna().sum(),
        'repeated_keys': valid_keys.duplicated().sum(),
        'exact_duplicate_rows': df.duplicated().sum(),
    })

display(pd.DataFrame(summary_rows).set_index('table'))

print('Число уникальных пользователей:')
display(pd.Series({
    'customers': customers_raw['customer_id'].nunique(),
    'sessions': sessions_raw['customer_id'].nunique(),
    'orders': orders_raw['customer_id'].nunique(),
}, name='unique_customers').to_frame())

print('Число уникальных сессий в событиях:')
print(events_raw['session_id'].nunique())

,rows,primary_key,unique_keys,missing_keys,repeated_keys,exact_duplicate_rows
table,,,,,,
customers,17846,customer_id,17846,0,0,0
sessions,37699,session_id,37699,0,0,0
events,239089,event_id,239089,0,0,0
orders,10652,order_id,10652,0,0,0


Число уникальных пользователей:


,unique_customers
customers,17846
sessions,16894
orders,8230


Число уникальных сессий в событиях:
37699


In [5]:
missing_rows = []

for name, df in raw_tables.items():
    for column in df.columns:
        missing_rows.append({
            'table': name,
            'column': column,
            'missing_count': df[column].isna().sum(),
            'missing_pct': df[column].isna().mean() * 100,
        })

missing_summary = pd.DataFrame(missing_rows)
display(missing_summary.round({'missing_pct': 2}))

,table,column,missing_count,missing_pct
0,customers,customer_id,0,0.00
1,customers,signup_date,0,0.00
2,sessions,session_id,0,0.00
3,sessions,customer_id,0,0.00
4,sessions,start_time,0,0.00
5,events,event_id,0,0.00
6,events,session_id,0,0.00
7,events,timestamp,0,0.00
8,events,event_type,0,0.00
9,events,product_id,24881,10.41


In [6]:
time_columns = {
    'customers': 'signup_date',
    'sessions': 'start_time',
    'events': 'timestamp',
    'orders': 'order_time',
}

date_rows = []

for name, column in time_columns.items():
    source = raw_tables[name][column]
    parsed = pd.to_datetime(source, errors='coerce', utc=True)

    date_rows.append({
        'table': name,
        'time_column': column,
        'min_date': parsed.min(),
        'max_date': parsed.max(),
        'missing_source': source.isna().sum(),
        'invalid_nonmissing': (source.notna() & parsed.isna()).sum(),
    })

date_summary = pd.DataFrame(date_rows).set_index('table')
display(date_summary)

,time_column,min_date,max_date,missing_source,invalid_nonmissing
table,,,,,
customers,signup_date,2020-01-01 00:00:00+00:00,2025-10-31 00:00:00+00:00,0,0
sessions,start_time,2024-01-01 00:57:40+00:00,2025-10-31 23:34:11+00:00,0,0
events,timestamp,2024-01-01 01:02:40+00:00,2025-11-01 01:50:04+00:00,0,0
orders,order_time,2024-01-01 07:08:06+00:00,2025-10-31 22:59:41+00:00,0,0


In [7]:
customer_ids = set(customers_raw['customer_id'].dropna())
session_ids = set(sessions_raw['session_id'].dropna())

relationship_checks = [
    ('sessions', 'customer_id', customer_ids, 'customers'),
    ('orders', 'customer_id', customer_ids, 'customers'),
    ('events', 'session_id', session_ids, 'sessions'),
]

relationship_rows = []

for table_name, column, parent_ids, parent_table in relationship_checks:
    values = raw_tables[table_name][column]

    relationship_rows.append({
        'table': table_name,
        'key': column,
        'parent_table': parent_table,
        'missing_key_rows': values.isna().sum(),
        'unmatched_rows': (
            values.notna() & ~values.isin(parent_ids)
        ).sum(),
    })

display(pd.DataFrame(relationship_rows))

,table,key,parent_table,missing_key_rows,unmatched_rows
0,sessions,customer_id,customers,0,0
1,orders,customer_id,customers,0,0
2,events,session_id,sessions,0,0


In [8]:
events_eda = (
    events_raw
    .dropna(subset=['event_id'])
    .drop_duplicates(subset='event_id')
    .copy()
)

funnel_order = ['page_view', 'add_to_cart', 'checkout', 'purchase']

funnel = (
    events_eda
    .groupby('event_type')
    .agg(
        event_count=('event_id', 'size'),
        session_count=('session_id', 'nunique'),
    )
    .reindex(funnel_order, fill_value=0)
)

previous_count = funnel['event_count'].shift(1)

funnel['event_ratio_to_previous'] = (
    funnel['event_count']
    .div(previous_count.where(previous_count.ne(0)))
)

display(funnel)

print('Все встречающиеся типы событий:')
display(events_eda['event_type'].value_counts(dropna=False).to_frame('count'))

print('Доля событий без product_id по типам:')
display(
    events_eda
    .assign(missing_product=events_eda['product_id'].isna())
    .groupby('event_type')['missing_product']
    .mean()
    .mul(100)
    .round(2)
    .to_frame('missing_product_pct')
)

,event_count,session_count,event_ratio_to_previous
event_type,,,
page_view,169300,37699,NaN
add_to_cart,44908,25601,0.265257
checkout,14229,14229,0.316848
purchase,10652,10652,0.748612


Все встречающиеся типы событий:


,count
event_type,
page_view,169300
add_to_cart,44908
checkout,14229
purchase,10652


Доля событий без product_id по типам:


,missing_product_pct
event_type,
add_to_cart,0.0
checkout,100.0
page_view,0.0
purchase,100.0


In [9]:
total_numeric = pd.to_numeric(
    orders_raw['total_usd'],
    errors='coerce',
)

print('Проверка total_usd:')
display(pd.Series({
    'missing_source': orders_raw['total_usd'].isna().sum(),
    'invalid_nonmissing': (
        orders_raw['total_usd'].notna() & total_numeric.isna()
    ).sum(),
    'negative_values': total_numeric.lt(0).sum(),
    'zero_values': total_numeric.eq(0).sum(),
}, name='count').to_frame())

display(total_numeric.describe().to_frame('total_usd'))

Проверка total_usd:


,count
missing_source,0
invalid_nonmissing,0
negative_values,0
zero_values,0


,total_usd
count,10652.000000
mean,133.176466
std,148.664093
min,2.800000
25%,40.267500
50%,86.055000
75%,174.677500
max,2680.640000


### Выводы по первой части

**Структура и качество данных.** Загружены четыре таблицы: `customers` — 17 846 пользователей, `sessions` — 37 699 сессий, `events` — 239 089 событий, `orders` — 10 652 заказа. Первичные ключи уникальны, пропусков в них и полных дубликатов нет. Все связи корректны: сессии и заказы связаны с существующими пользователями, события — с существующими сессиями. Получить `customer_id` для событий можно через объединение с `sessions` по `session_id`, без увеличения числа строк.

**Покрытие пользователей.** В таблице сессий представлены 16 894 пользователя, в таблице заказов — 8 230 покупателей. Поэтому итоговый набор пользователей нельзя формировать только по активности или покупкам: пользователи без соответствующей истории потеряются. Рассчитанные признаки будем присоединять к выбранному набору пользователей через левое объединение, заполняя отсутствующие значения по правилам задания.

**Временные границы.** Даты регистрации находятся в диапазоне от 1 января 2020 года до 31 октября 2025 года. Сессии и заказы охватывают период с 1 января 2024 года по 31 октября 2025 года, события — до 1 ноября 2025 года. Пропусков и ошибок преобразования временных полей не обнаружено. Для пробного расчёта выбираем `run_date = '2025-09-01'`: доступная история охватывает оба окна агрегации — 7 и 30 дней — и последующий семидневный период.

**Пропуски в товарных идентификаторах.** Единственное поле с пропусками — `events.product_id`: 24 881 строка, или 10,41% событий. Идентификатор товара отсутствует у всех событий `checkout` и `purchase`, но заполнен у всех `page_view` и `add_to_cart`. Такие строки исключаем только из расчёта числа уникальных товаров; для остальных показателей сохраняем.

**Воронка покупки.** В данных представлены все четыре ожидаемых типа событий: 169 300 просмотров страниц, 44 908 добавлений в корзину, 14 229 переходов к оплате и 10 652 покупки. Отношение `add_to_cart / page_view` по всей истории составляет 26,53%, а `purchase / add_to_cart` — 23,72%. Эти показатели отражают отношения числа событий, а не конверсии уникальных пользователей. Число событий `purchase` совпадает с числом заказов, однако само совпадение не доказывает соответствие отдельных записей.

**Денежные показатели.** Все значения `total_usd` успешно преобразуются в числа; пропусков, отрицательных и нулевых сумм нет. Средняя сумма заказа — 133,18 USD, медиана — 86,06 USD, минимум — 2,80 USD, максимум — 2 680,64 USD. Среднее выше медианы из-за влияния крупных заказов. Оснований исключать их на текущем этапе нет. Денежные признаки рассчитываем по таблице `orders`.

**Ограничения для расчёта.** Все источники необходимо фильтровать по собственным временным полям, исключая записи в момент `run_date` и позже. В частности, начало сессии до даты среза не позволяет использовать её будущие события при расчёте длительности. Давность последней покупки определяется по всей доступной истории заказов до среза; значение `−1` означает отсутствие заказов в этой истории.

**Итог.** Структура данных, связи и основные особенности изучены. Можно переходить к предобработке и расчёту 17 батч-признаков на выбранную дату. Перед расчётом нужно зафиксировать часовой пояс исходных данных и привести временные поля к единому стандарту.

# **Часть 2. Подготовка batch-признаков**


#### **Задание 2.1: подготовьте данные**

In [10]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT table_name, column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND (
                  (table_name = 'customers' AND column_name = 'signup_date')
                  OR (table_name = 'sessions' AND column_name = 'start_time')
                  OR (table_name = 'events' AND column_name = 'timestamp')
                  OR (table_name = 'orders' AND column_name = 'order_time')
              )
            ORDER BY table_name
        """),
    )

    time_schema = pd.DataFrame(result.fetchall(), columns=list(result.keys()))
display(time_schema)

,table_name,column_name,data_type
0,customers,signup_date,date
1,events,timestamp,timestamp without time zone
2,orders,order_time,timestamp without time zone
3,sessions,start_time,timestamp without time zone


In [11]:
RUN_DATE = batch_module.normalize_run_date('2025-09-01')
prepared = batch_module.preprocess_tables(raw_tables, RUN_DATE)
customers_snapshot = prepared['customers']
sessions_before_t = prepared['sessions']
events_before_t = prepared['events']
orders_before_t = prepared['orders']
eligible_customer_ids = customers_snapshot['customer_id']

# Проверки предобработки и отсутствия будущего выполняются независимо.
for name, column in [('customers', 'signup_date'), ('sessions', 'start_time'),
                     ('events', 'timestamp'), ('orders', 'order_time'),
                     ('last_orders', 'last_order_time')]:
    assert prepared[name][column].lt(RUN_DATE).all()
assert events_before_t['start_time'].lt(RUN_DATE).all()
assert events_before_t['customer_id'].isin(eligible_customer_ids).all()
print(f'Дата среза: {RUN_DATE}')
print('Проверки предобработки пройдены.')
display(pd.Series({k: len(v) for k, v in prepared.items()}, name='rows').to_frame())


Дата среза: 2025-09-01 00:00:00+00:00
Проверки предобработки пройдены.


,rows
customers,17292
sessions,33205
events,210246
orders,9348
last_orders,7404


#### **Задание 2.2: рассчитайте признаки**

In [12]:
# Актуальные формулы и правила находятся только в Python-модуле.
features = batch_module.build_batch_features(raw_tables, RUN_DATE)
FEATURE_COLUMNS = batch_module.FEATURE_COLUMNS
OUTPUT_COLUMNS = batch_module.OUTPUT_COLUMNS
count_columns = [c for c in FEATURE_COLUMNS if c.startswith((
    'page_view_count', 'add_to_cart_count', 'unique_products',
    'session_count', 'order_count'))]
conversion_columns = [c for c in FEATURE_COLUMNS if 'conversion' in c]
orders_30d = orders_before_t.loc[
    orders_before_t['order_time'].ge(RUN_DATE - pd.Timedelta(days=30))
]
print(f'Итоговая таблица: {features.shape}')
display(features.head(10))
display(features[FEATURE_COLUMNS].describe().T)


Итоговая таблица: (17292, 19)


,customer_id,run_date,page_view_count_7d,page_view_count_30d,add_to_cart_count_7d,add_to_cart_count_30d,view_to_cart_conversion_7d,view_to_cart_conversion_30d,cart_to_purchase_conversion_7d,cart_to_purchase_conversion_30d,unique_products_7d,unique_products_30d,avg_session_duration_30d,session_count_7d,session_count_30d,days_since_last_purchase,order_count_30d,total_spend_30d,avg_order_value_30d
0,1,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,67,0,0.0,0.0
1,2,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,241,0,0.0,0.0
2,3,2025-09-01 00:00:00+00:00,8,8,0,0,0.0,0.0,0.0,0.0,8,8,7920.0,1,1,-1,0,0.0,0.0
3,4,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
4,5,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,213,0,0.0,0.0
5,6,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,277,0,0.0,0.0
6,7,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
7,9,2025-09-01 00:00:00+00:00,0,4,0,0,0.0,0.0,0.0,0.0,0,4,2400.0,0,1,467,0,0.0,0.0
8,10,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,-1,0,0.0,0.0
9,12,2025-09-01 00:00:00+00:00,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0.0,0,0,392,0,0.0,0.0


,count,mean,std,min,25%,50%,75%,max
page_view_count_7d,17292.0,0.092471,0.717727,0.0,0.0,0.0,0.0,11.00
page_view_count_30d,17292.0,0.441302,1.559259,0.0,0.0,0.0,0.0,15.00
add_to_cart_count_7d,17292.0,0.026024,0.238832,0.0,0.0,0.0,0.0,5.00
add_to_cart_count_30d,17292.0,0.119130,0.515033,0.0,0.0,0.0,0.0,7.00
view_to_cart_conversion_7d,17292.0,0.006066,0.056380,0.0,0.0,0.0,0.0,1.00
view_to_cart_conversion_30d,17292.0,0.025641,0.111655,0.0,0.0,0.0,0.0,1.00
cart_to_purchase_conversion_7d,17292.0,0.004972,0.065158,0.0,0.0,0.0,0.0,1.00
cart_to_purchase_conversion_30d,17292.0,0.019618,0.125651,0.0,0.0,0.0,0.0,1.00
unique_products_7d,17292.0,0.092124,0.714865,0.0,0.0,0.0,0.0,11.00
unique_products_30d,17292.0,0.438931,1.550576,0.0,0.0,0.0,0.0,15.00


In [13]:
# Состав, уникальность и сохранение пользователей
assert len(FEATURE_COLUMNS) == 17
assert features.columns.tolist() == OUTPUT_COLUMNS
assert not features.duplicated(['customer_id', 'run_date']).any()
assert len(features) == len(customers_snapshot)

assert set(features['customer_id']) == set(customers_snapshot['customer_id'])
assert features['run_date'].eq(RUN_DATE).all()

# Пропуски, бесконечности и диапазоны
assert not features.isna().any().any()
assert np.isfinite(features[FEATURE_COLUMNS].to_numpy(dtype=float)).all()
assert features[count_columns].ge(0).all().all()
assert features[conversion_columns].ge(0).all().all()

nonnegative_float_columns = [
    'avg_session_duration_30d',
    'total_spend_30d',
    'avg_order_value_30d',
]
assert features[nonnegative_float_columns].ge(0).all().all()
assert features['days_since_last_purchase'].ge(-1).all()

# Вложенность окон: счётчики за 7 дней не больше счётчиков за 30 дней
for prefix in [
    'page_view_count',
    'add_to_cart_count',
    'unique_products',
    'session_count',
]:
    assert features[f'{prefix}_7d'].le(features[f'{prefix}_30d']).all()

# Нулевые знаменатели должны давать нулевые конверсии
for days in [7, 30]:
    no_views = features[f'page_view_count_{days}d'].eq(0)
    no_cart = features[f'add_to_cart_count_{days}d'].eq(0)

    assert features.loc[
        no_views, f'view_to_cart_conversion_{days}d'
    ].eq(0.0).all()

    assert features.loc[
        no_cart, f'cart_to_purchase_conversion_{days}d'
    ].eq(0.0).all()

# Пользователи без заказов
no_order_history = ~features['customer_id'].isin(
    orders_before_t['customer_id']
)
assert features.loc[
    no_order_history, 'days_since_last_purchase'
].eq(-1).all()

no_recent_orders = features['order_count_30d'].eq(0)
assert features.loc[
    no_recent_orders, ['total_spend_30d', 'avg_order_value_30d']
].eq(0.0).all().all()

# Сверка итоговых денежных агрегатов с источником
assert features['order_count_30d'].sum() == orders_30d['order_id'].nunique()

assert np.isclose(
    features['total_spend_30d'].sum(),
    orders_30d['total_usd'].sum(),
)

has_recent_orders = ~no_recent_orders
assert np.allclose(
    features.loc[has_recent_orders, 'avg_order_value_30d'],
    (
        features.loc[has_recent_orders, 'total_spend_30d']
        / features.loc[has_recent_orders, 'order_count_30d']
    ),
)

# Сверка событийных счётчиков с источником
for days in [7, 30]:
    window_events = events_before_t.loc[
        events_before_t['timestamp'].ge(
            RUN_DATE - pd.Timedelta(days=days)
        )
    ]

    for event_type, prefix in [
        ('page_view', 'page_view_count'),
        ('add_to_cart', 'add_to_cart_count'),
    ]:
        assert features[f'{prefix}_{days}d'].sum() == (
            window_events['event_type'].eq(event_type).sum()
        )

# Явная проверка отсутствия будущих записей
assert events_before_t['timestamp'].lt(RUN_DATE).all()
assert sessions_before_t['start_time'].lt(RUN_DATE).all()
assert orders_before_t['order_time'].lt(RUN_DATE).all()

print('Все проверки итоговой таблицы пройдены.')

Все проверки итоговой таблицы пройдены.


In [14]:
# Проверяем функцию даже если нужного случая нет в реальных данных.
np.testing.assert_array_equal(
    batch_module.safe_ratio([3, 0, 2], [0, 0, 4]), [0.0, 0.0, 0.5]
)
print('Проверка безопасного деления пройдена.')
display(features[conversion_columns].gt(1).sum().to_frame('Конверсий больше 1'))


Проверка безопасного деления пройдена.


,Конверсий больше 1
view_to_cart_conversion_7d,0
view_to_cart_conversion_30d,0
cart_to_purchase_conversion_7d,0
cart_to_purchase_conversion_30d,0


### Выводы по второй части

**Единая логика.** Предобработка и расчёт 17 признаков выполняются модулем `calculate_batch_features.py`. В тетради остаются исследование, вызовы модуля и независимые проверки.

**Дата среза.** Пользователи отбираются по `signup_date < run_date` при каждом запуске. Фиксированной когорты и запрета ранних дат нет. Окна имеют границы `[T − 7 дней, T)` и `[T − 30 дней, T)`.

**Обработка данных.** Исходное время без часового пояса трактуется как UTC. Дубликаты событий удаляются по `event_id`; невалидные даты исключаются. Строки без `product_id` не участвуют только в подсчёте уникальных товаров. Счётчики заполняются нулями, вещественные признаки — `0.0`, давность покупки при отсутствии заказов — `−1`. При нулевом знаменателе конверсия равна `0.0`.

**Загрузка.** DAG читает события и заказы за 30 дней, а последние покупки получает через `MAX(order_time)`. Сессии старше окна загружаются только как контекст событий в окне. Полная история в первой части нужна исключительно для EDA и проверки эквивалентности.

**Проверки.** Ячейки выше проверяют уникальность ключа, состав колонок, временные ограничения, пропуски, деление на ноль и сверку агрегатов с источниками. Ниже предусмотрено сравнение полной и оконной загрузки на четырёх датах, а после повторных прогонов DAG — сравнение локальных результатов с CSV из S3.

**Общий набор пользователей.** Полные срезы включают всех пользователей, доступных на соответствующую дату. Для конкретной обучающей выборки и сравнения с инференсом общий набор формируется отдельно пересечением `customer_id`; новые пользователи не удаляются из полного инференсного файла.

**Статус после доработки.** Выводы выполнения очищены, чтобы не показывать результаты прежней версии как результаты исправленной. Перед повторной сдачей необходимо выполнить Restart & Run All и повторные запуски DAG, сохранив актуальные результаты.


In [15]:
# Один модуль используется и для полной истории, и для оконной SQL-загрузки.
# Сравниваем два маршрута чтения, а не две копии бизнес-логики.
check_dates = ['2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01']
expected_rows = {'2025-08-01': 16992, '2025-09-01': 17292,
                 '2025-10-01': 17558, '2025-11-01': 17846}
local_slices = {}
comparison_rows = []
for date in check_dates:
    full_result = batch_module.build_batch_features(raw_tables, date)
    window_tables = batch_module.load_source_tables(connection_url, date)
    optimized_result = batch_module.build_batch_features(window_tables, date)
    for df in [full_result, optimized_result]:
        df['run_date'] = df['run_date'].astype('datetime64[ns, UTC]')
    pd.testing.assert_frame_equal(
        full_result, optimized_result,
        check_exact=False, rtol=1e-10, atol=1e-10,
    )
    assert len(optimized_result) == expected_rows[date]
    local_slices[date] = optimized_result
    comparison_rows.append({
        'run_date': date, 'users': len(optimized_result),
        'loaded_events': len(window_tables['events']),
        'loaded_orders': len(window_tables['orders']),
    })
display(pd.DataFrame(comparison_rows))
print('На всех четырёх датах оконная SQL-загрузка совпала с полной историей.')


,run_date,users,loaded_events,loaded_orders
0,2025-08-01,16992,10581,453
1,2025-09-01,17292,11172,507
2,2025-10-01,17558,11439,514
3,2025-11-01,17846,10693,474


На всех четырёх датах оконная SQL-загрузка совпала с полной историей.


In [16]:
print('Python:', sys.version.split()[0])

for package in ['pandas', 'numpy', 'SQLAlchemy', 'psycopg2-binary']:
    try:
        print(f'{package}: {version(package)}')
    except PackageNotFoundError:
        print(f'{package}: не установлен')
        

Python: 3.13.9
pandas: 2.3.3
numpy: 2.3.5
SQLAlchemy: 2.0.43
psycopg2-binary: 2.9.12


In [17]:
# Выполнить после обновления DAG и повторных запусков на четырёх датах.
# В .env установите RUN_S3_CHECK=true, затем перезапустите ядро.
if os.getenv('RUN_S3_CHECK', 'false').lower() == 'true':
    required_s3 = ['S3_BUCKET', 'S3_ENDPOINT_URL', 'S3_ACCESS_KEY_ID', 'S3_SECRET_ACCESS_KEY']
    missing = [k for k in required_s3 if not os.getenv(k) or os.getenv(k).startswith('<')]
    if missing:
        raise ValueError(f'Заполните .env: {missing}')
    client = boto3.client(
        's3', endpoint_url=os.environ['S3_ENDPOINT_URL'],
        region_name=os.getenv('S3_REGION', 'ru-central1'),
        aws_access_key_id=os.environ['S3_ACCESS_KEY_ID'],
        aws_secret_access_key=os.environ['S3_SECRET_ACCESS_KEY'],
        config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
    )
    saved_slices = {}
    for date in check_dates:
        response = client.get_object(
            Bucket=os.environ['S3_BUCKET'],
            Key=f'run_date={date}/batch_features.csv',
        )
        try:
            df = pd.read_csv(BytesIO(response['Body'].read()))
        finally:
            response['Body'].close()
        df['run_date'] = pd.to_datetime(df['run_date'], utc=True).astype('datetime64[ns, UTC]')
        batch_module.validate_features(df, date)
        pd.testing.assert_frame_equal(
            local_slices[date].sort_values('customer_id').reset_index(drop=True),
            df.sort_values('customer_id').reset_index(drop=True),
            check_exact=False, rtol=1e-10, atol=1e-10,
        )
        saved_slices[date] = df
        print(f'{date}: {df.shape}, CSV из S3 совпадает с локальным расчётом')

    # Общий набор нужен на этапе подготовки выборки, а не в универсальном DAG.
    historical = saved_slices['2025-09-01']
    inference = saved_slices['2025-10-01']
    common_ids = set(historical.customer_id) & set(inference.customer_id)
    historical_aligned = historical.loc[historical.customer_id.isin(common_ids)]
    inference_aligned = inference.loc[inference.customer_id.isin(common_ids)]
    assert set(historical_aligned.customer_id) == set(inference_aligned.customer_id)
    assert len(common_ids) == 17292
    print(f'Для обучения и инференса согласовано {len(common_ids)} пользователей.')
    print('Новые пользователи сохранены в полном инференсном срезе.')
else:
    print('Проверка S3 пока не выполнялась: после прогонов задайте RUN_S3_CHECK=true.')


Проверка S3 пока не выполнялась: после прогонов задайте RUN_S3_CHECK=true.
